# 🎓 ProctorAI - Báo Cáo Kỹ Thuật: Phát Hiện Khuôn Mặt & Ước Lượng Hướng Đầu
> **Dự án Capstone:** Samsung Innovation Campus (SIC)  
> **Thành viên báo cáo:** Hiếu (AI & Computer Vision) & Rin (Pipeline & State)  
> **Chủ đề thực nghiệm:** Bóc tách từng bước (Step-by-Step) thuật toán MediaPipe Face Mesh, Mô hình 3D Nhân Trắc Học, Thuật toán giải phối cảnh `solvePnP` SQPNP và Cơ chế Hysteresis khi quay đầu.

---
### 📌 Mục Lục Thực Nghiệm
1. **Khởi tạo môi trường & Cấu hình hệ thống**
2. **Trích xuất 468 điểm mốc khuôn mặt (MediaPipe Face Mesh) & Đếm người**
3. **Mô hình 3D Nhân trắc học & 6 mốc giải phẫu cốt lõi**
4. **Giải thuật phối cảnh toàn cục SQPNP & Phân rã RQ Euler Angles**
5. **Chiếu trục tọa độ 3D (Pitch, Yaw, Roll) lên khuôn mặt**
6. **Kiểm chứng máy trạng thái Hysteresis (Nhớ trạng thái khi quay hẳn ra sau)**
7. **Demo tương tác Webcam thời gian thực (Interactive Real-time Demo)**


In [ ]:
# BƯỚC 1: IMPORT THƯ VIỆN & THIẾT LẬP MÔI TRƯỜNG
import sys
import os
import time
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt

# Đảm bảo import được module nội bộ từ src/
sys.path.insert(0, os.path.abspath("."))
from src.config import get_config
from src.head_pose import HeadPoseEstimator

cfg = get_config()
print("✅ Cấu hình hệ thống nạp thành công:")
print(f"   • Độ phân giải Camera: {cfg.camera.width}x{cfg.camera.height} @ {cfg.camera.fps_limit} FPS")
print(f"   • Ngưỡng quay trái/phải: {cfg.head_pose.yaw_side_threshold}°")
print(f"   • Ngưỡng quay hẳn ra sau: {cfg.head_pose.yaw_backward_threshold}°")
print(f"   • Ngưỡng cúi/ngửa mặt: {cfg.head_pose.pitch_threshold}°")
print(f"   • Ngưỡng xác nhận người thứ 2: {cfg.rules.multiple_faces_threshold_seconds}s")


---
## 📷 Bước 2: Thu Nhận Khung Hình Mẫu (Frame Ingestion)
Ở bước này, chúng ta sẽ bắt 1 khung hình từ Webcam để làm mẫu thử nghiệm xuyên suốt các bước tiếp theo.
*(Nếu máy không cắm webcam hoặc camera đang bận, code tự động sinh một ảnh chân dung nhân tạo để bạn vẫn chạy demo bình thường).*


In [ ]:
# BƯỚC 2: CHỤP 1 FRAME MẪU TỪ WEBCAM HOẶC SINH KHUNG HÌNH MẪU
def get_sample_frame():
    cap = cv2.VideoCapture(cfg.camera.device_id)
    frame = None
    if cap.isOpened():
        # Đọc vài frame đầu để camera tự cân bằng sáng
        for _ in range(5):
            ret, tmp = cap.read()
            if ret:
                frame = tmp
        cap.release()
    
    if frame is None:
        print("⚠️ Không mở được webcam vật lý, tạo ảnh nhân tạo phục vụ mô phỏng...")
        frame = np.full((480, 640, 3), 40, dtype=np.uint8)
        # Vẽ một khuôn mặt giả lập
        cv2.ellipse(frame, (320, 240), (120, 160), 0, 0, 360, (200, 180, 160), -1)
        cv2.circle(frame, (270, 200), 12, (50, 50, 50), -1) # Mắt trái
        cv2.circle(frame, (370, 200), 12, (50, 50, 50), -1) # Mắt phải
        cv2.circle(frame, (320, 245), 8, (120, 100, 90), -1) # Mũi
        cv2.ellipse(frame, (320, 310), (45, 15), 0, 0, 180, (80, 60, 180), -1) # Miệng
    else:
        frame = cv2.flip(frame, 1) # Lật gương tự nhiên
        print("✅ Đã chụp thành công 1 frame thật từ Webcam!")
        
    return frame

sample_bgr = get_sample_frame()
sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 5))
plt.imshow(sample_rgb)
plt.title(f"Khung hình mẫu ({sample_bgr.shape[1]}x{sample_bgr.shape[0]})", fontsize=12, fontweight='bold')
plt.axis("off")
plt.show()


---
## 🕸️ Bước 3: MediaPipe Face Mesh & Phát Hiện Người Thứ Hai
**Nguyên lý:**
1. **MediaPipe Face Mesh:** Mô hình học sâu cực nhẹ (On-device) trích xuất **468 mốc không gian** trên khuôn mặt thí sinh theo thời gian thực.
2. **Đếm số người (`num_faces`):** 
   * `len(results.multi_face_landmarks) == 0`: Thí sinh mất mặt hoặc che camera.
   * `len(results.multi_face_landmarks) > 1`: Phát hiện có từ 2 người trở lên trong phòng thi (gian lận nghiêm trọng).
   * `len(results.multi_face_landmarks) == 1`: Phòng thi hợp lệ (1 thí sinh), tiến hành tính góc đầu.


In [ ]:
# BƯỚC 3: PHÂN TÍCH FACEMESH & TRỰC QUAN HÓA TOÀN BỘ 468 MỐC
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=4,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

results = face_mesh.process(sample_rgb)
num_faces = len(results.multi_face_landmarks) if results.multi_face_landmarks else 0
print(f"📊 Kết quả đếm khuôn mặt: {num_faces} người")

annotated_mesh = sample_rgb.copy()
if results.multi_face_landmarks:
    for face_landmarks in results.multi_face_landmarks:
        mp_drawing.draw_landmarks(
            image=annotated_mesh,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )

plt.figure(figsize=(8, 6))
plt.imshow(annotated_mesh)
plt.title(f"MediaPipe Face Mesh (468 mốc) - Phát hiện: {num_faces} khuôn mặt", fontsize=12, fontweight='bold', color='navy')
plt.axis("off")
plt.show()


---
## 📐 Bước 4: Mô Hình Nhân Trắc Học 3D & 6 Điểm Mốc Giải Phẫu Cốt Lõi
Thay vì dùng toàn bộ 468 điểm làm chậm quá trình tính toán, hệ thống áp dụng **Mô hình Nhân trắc học 3D chuẩn (Canonical 3D Model)** với đúng 6 điểm mốc giải phẫu bất biến:

| Điểm mốc giải phẫu | Index MediaPipe | Tọa độ 3D chuẩn $(X, Y, Z)$ (mm) | Vai trò hình học |
| :--- | :---: | :---: | :--- |
| **Đỉnh chóp mũi** | `1` | `(0.0, 0.0, 0.0)` | Gốc tọa độ, điểm nhô cao nhất |
| **Đáy cằm** | `199` | `(0.0, 110.0, 65.0)` | Định vị trục dọc cằm (Pitch) |
| **Khóe mắt ngoài trái** | `33` | `(-65.0, -50.0, 60.0)` | Cố định trục ngang mắt trái (Yaw) |
| **Khóe mắt ngoài phải** | `263` | `(65.0, -50.0, 60.0)` | Cố định trục ngang mắt phải (Yaw) |
| **Khóe miệng ngoài trái** | `61` | `(-40.0, 50.0, 40.0)` | Định vị mặt phẳng miệng |
| **Khóe miệng ngoài phải** | `291` | `(40.0, 50.0, 40.0)` | Định vị mặt phẳng miệng |

### Ma trận nội thông số Camera (Camera Intrinsic Matrix $K$):
$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix} = \begin{bmatrix} W & 0 & W/2 \\ 0 & W & H/2 \\ 0 & 0 & 1 \end{bmatrix}$$


In [ ]:
# BƯỚC 4: TRÍCH XUẤT 6 MỐC GIẢI PHẪU & TRỰC QUAN HÓA SO SÁNH
FACE_KEYPOINTS = [1, 199, 33, 263, 61, 291]
KEYPOINT_LABELS = {
    1: "1: Đỉnh mũi",
    199: "199: Đáy cằm",
    33: "33: Mắt trái",
    263: "263: Mắt phải",
    61: "61: Miệng trái",
    291: "291: Miệng phải"
}

h, w, _ = sample_bgr.shape
pts_2d = []

annotated_6pts = sample_rgb.copy()

if results.multi_face_landmarks:
    landmarks = results.multi_face_landmarks[0].landmark
    for idx in FACE_KEYPOINTS:
        px = int(landmarks[idx].x * w)
        py = int(landmarks[idx].y * h)
        pts_2d.append([px, py])
        # Vẽ điểm đỏ nổi bật
        cv2.circle(annotated_6pts, (px, py), 6, (255, 0, 0), -1)
        cv2.circle(annotated_6pts, (px, py), 8, (255, 255, 255), 2)
        cv2.putText(annotated_6pts, str(idx), (px + 10, py - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)

pts_2d = np.array(pts_2d, dtype=np.float64)

plt.figure(figsize=(8, 6))
plt.imshow(annotated_6pts)
plt.title("6 Mốc Giải Phẫu Cốt Lõi Được Trích Xuất (Canonical Keypoints)", fontsize=12, fontweight='bold')
plt.axis("off")
plt.show()

print("Tọa độ 2D tương ứng trên ảnh camera (Pixel):")
for i, idx in enumerate(FACE_KEYPOINTS):
    print(f"  • {KEYPOINT_LABELS[idx]:<18} : ({pts_2d[i][0]:.1f}, {pts_2d[i][1]:.1f}) px")


---
## 🧠 Bước 5: Giải Thuật SQPNP & Phân Rã Ma Trận RQ Thành Góc Euler
### 1. Tại sao dùng `SOLVEPNP_SQPNP` thay vì `SOLVEPNP_ITERATIVE`?
* **Phương pháp cũ (`ITERATIVE` - Levenberg-Marquardt):** Rất dễ kẹt vào điểm cực tiểu cục bộ (local minima), gây ra hiện tượng nhảy góc $180^\circ$ khi thí sinh quay đầu nhanh.
* **Phương pháp mới (`SQPNP` - Standard Quadratic PnP):** Giải bài toán tối ưu toàn phương không lặp với độ phức tạp $O(N)$, bảo đảm tìm ra nghiệm toàn cục chuẩn xác, ổn định tuyệt đối.

### 2. Thuật toán phân rã RQ (`cv2.RQDecomp3x3`):
Sau khi giải PnP tìm vector quay $r_{vec}$, hàm `cv2.Rodrigues` chuyển thành ma trận quay $R$ ($3\times3$).  
Thuật toán phân rã RQ phân rã $R$ thành tích của 3 ma trận xoay chuẩn theo các trục tọa độ:
$$R = R_x(\text{Pitch}) \cdot R_y(\text{Yaw}) \cdot R_z(\text{Roll})$$

> **⚠️ Lưu ý sống còn để trả lời câu hỏi của thầy:**  
> Hàm `cv2.RQDecomp3x3` của OpenCV **đã trả về kết quả trực tiếp theo đơn vị Độ ($^\circ$)**.  
> Nhiều bài hướng dẫn trên mạng mắc lỗi nhân thêm `* 360` làm góc bị thổi phồng hàng ngàn độ. ProctorAI đã **loại bỏ hoàn toàn hệ số nhân sai này**, giúp góc đo chính xác đến từng số thập phân!


In [ ]:
# BƯỚC 5: TÍNH TOÁN GÓC PITCH, YAW, ROLL VỚI SQPNP
MODEL_POINTS_3D = np.array([
    (0.0, 0.0, 0.0),             # 1: Đỉnh mũi
    (0.0, 110.0, 65.0),          # 199: Đáy cằm
    (-65.0, -50.0, 60.0),        # 33: Khóe mắt trái
    (65.0, -50.0, 60.0),         # 263: Khóe mắt phải
    (-40.0, 50.0, 40.0),         # 61: Khóe miệng trái
    (40.0, 50.0, 40.0)           # 291: Khóe miệng phải
], dtype=np.float64)

cam_matrix = np.array([
    [w, 0, w / 2],
    [0, w, h / 2],
    [0, 0, 1]
], dtype=np.float64)
dist_coeffs = np.zeros((4, 1), dtype=np.float64)

# Giải SQPNP
success, rot_vec, trans_vec = cv2.solvePnP(
    MODEL_POINTS_3D,
    pts_2d,
    cam_matrix,
    dist_coeffs,
    flags=cv2.SOLVEPNP_SQPNP
)

# Chuyển đổi sang Ma trận quay R và phân rã Euler
rot_mat, _ = cv2.Rodrigues(rot_vec)
angles, _, _, _, _, _ = cv2.RQDecomp3x3(rot_mat)

pitch = round(float(angles[0]), 2)
yaw = round(float(angles[1]), 2)
roll = round(float(angles[2]), 2)

print("=" * 50)
print("📊 KẾT QUẢ ĐO GÓC ĐẦU BẰNG SQPNP:")
print(f"   • Pitch (Cúi / Ngửa) : {pitch:+.2f}°  (Dương: Cúi gằm, Âm: Ngửa lên)")
print(f"   • Yaw   (Trái / Phải): {yaw:+.2f}°  (Dương: Quay trái, Âm: Quay phải)")
print(f"   • Roll  (Nghiêng vai): {roll:+.2f}°  (Nghiêng đầu sang vai)")
print("=" * 50)


---
## 🎯 Bước 6: Chiếu Trục Tọa Độ 3D (3D Pose Axes) Lên Khuôn Mặt
Để trực quan hóa góc nhìn của thí sinh cho giám thị, hệ thống dùng `cv2.projectPoints` chiếu 3 trục từ đỉnh mũi ra màn hình 2D:
* 🔴 **Trục X (Màu đỏ):** Hướng sang ngang bên phải (Trục quay Pitch).
* 🟢 **Trục Y (Màu xanh lá):** Hướng thẳng xuống dưới (Trục quay Yaw).
* 🔵 **Trục Z (Màu xanh dương):** Hướng vuông góc từ chóp mũi đâm thẳng ra ngoài (Vector hướng nhìn chính của thí sinh).


In [ ]:
# BƯỚC 6: CHIẾU 3 TRỤC TỌA ĐỘ VÀ HIỂN THỊ HƯỚNG NHÌN
axis_length = 80.0
axis_3d = np.array([
    (axis_length, 0, 0),    # Trục X (Đỏ)
    (0, axis_length, 0),    # Trục Y (Xanh lá)
    (0, 0, axis_length)     # Trục Z (Xanh dương - Vector nhìn)
], dtype=np.float64)

# Chiếu 3D sang 2D pixel
axis_2d, _ = cv2.projectPoints(axis_3d, rot_vec, trans_vec, cam_matrix, dist_coeffs)

nose_pt = (int(pts_2d[0][0]), int(pts_2d[0][1]))
x_pt = (int(axis_2d[0][0][0]), int(axis_2d[0][0][1]))
y_pt = (int(axis_2d[1][0][0]), int(axis_2d[1][0][1]))
z_pt = (int(axis_2d[2][0][0]), int(axis_2d[2][0][1]))

annotated_axes = sample_rgb.copy()
# Vẽ 3 trục tọa độ từ đỉnh mũi
cv2.line(annotated_axes, nose_pt, x_pt, (255, 0, 0), 3)   # X: Đỏ
cv2.line(annotated_axes, nose_pt, y_pt, (0, 255, 0), 3)   # Y: Xanh lá
cv2.line(annotated_axes, nose_pt, z_pt, (0, 100, 255), 4) # Z: Xanh dương (Hướng nhìn)
cv2.circle(annotated_axes, nose_pt, 5, (255, 255, 0), -1)

# Phân loại trạng thái vi phạm
status = "BÌNH THƯỜNG - HỢP LỆ"
if abs(yaw) > cfg.head_pose.yaw_backward_threshold:
    status = "CẢNH BÁO: QUAY HẲN RA SAU"
elif yaw > cfg.head_pose.yaw_side_threshold:
    status = "CHÚ Ý: QUAY SANG TRÁI"
elif yaw < -cfg.head_pose.yaw_side_threshold:
    status = "CHÚ Ý: QUAY SANG PHẢI"
elif pitch > cfg.head_pose.pitch_threshold:
    status = "CHÚ Ý: CÚI ĐẦU XUỐNG GẦM BÀN"
elif pitch < -cfg.head_pose.pitch_threshold:
    status = "CHÚ Ý: NGỬA MẶT LÊN TRẦN"

plt.figure(figsize=(9, 6))
plt.imshow(annotated_axes)
plt.title(f"3D Head Pose Axes Projection\nPitch: {pitch}° | Yaw: {yaw}° | Trạng thái: {status}", 
          fontsize=12, fontweight='bold', color='darkred' if "CẢNH BÁO" in status else 'darkgreen')
plt.axis("off")
plt.show()


---
## 🔄 Bước 7: Cơ Chế Hysteresis (Nhớ Trạng Thái Khi Quay Hẳn Ra Sau)
### 1. Vấn đề thực tế:
MediaPipe Face Mesh là mô hình nhận diện khuôn mặt phía trước. Khi thí sinh quay đầu ngoái hẳn ra phía sau ($>35^\circ$ đến $90^\circ$), camera chỉ nhìn thấy phần **gáy**, không còn mốc khuôn mặt nào.  
Nếu không có bộ nhớ trạng thái, hệ thống sẽ nhầm lẫn đây là hành vi `"Mất mặt / Che camera"`.

### 2. Thuật toán chuyển tiếp Hysteresis:
```text
           [ Khuôn mặt đang được theo dõi ]
                         │
                         ▼
             abs(yaw) >= 35.0 độ ?
               ├─► [ĐÚNG]: Bật cờ turned_backward = True
               └─► [SAI]:  Nếu abs(yaw) <= 22.0 độ -> turned_backward = False
                         │
                         ▼
      [ Thí sinh tiếp tục ngoái đầu làm mất mốc 2D ]
                         │
                         ▼
             turned_backward đang BẬT?
               ├─► [ĐÚNG]: Báo "Quay hẳn ra sau" (Duy trì vi phạm)
               └─► [SAI]:  Báo "Mất mặt / Che camera"
```


In [ ]:
# BƯỚC 7: KIỂM CHỨNG MÔ HÌNH HYSTERESIS BẰNG HEADPOSEESTIMATOR
estimator = HeadPoseEstimator(cam_w=w, cam_h=h)

print("🧪 BẮT ĐẦU CHUỖI KIỂM THỬ TRẠNG THÁI XOAY ĐẦU:")

# Kịch bản 1: Thí sinh ngồi thẳng
pitch, yaw, viol = estimator.process(sample_rgb)
print(f"1. Thí sinh ngồi thẳng       : Pitch={pitch}°, Yaw={yaw}° -> Trạng thái: {viol or 'Bình thường'}")

# Kịch bản 2: Giả lập góc quay lớn (Yaw = 38.0° > 35.0°)
estimator.turned_backward = True
estimator.last_pitch = 4.0
estimator.last_yaw = 38.0
print(f"2. Thí sinh quay đầu > 35°   : Bật cờ turned_backward = {estimator.turned_backward}")

# Kịch bản 3: Thí sinh ngoái hẳn ra sau làm mất dấu mặt hoàn toàn (khung hình đen)
black_frame = np.zeros((h, w, 3), dtype=np.uint8)
pitch, yaw, viol = estimator.process(black_frame)
print(f"3. Khuất mặt hoàn toàn (gáy) : Pitch={pitch}°, Yaw={yaw}° -> Trạng thái: {viol}")
assert viol == "Quay hẳn ra sau", "Lỗi: Không duy trì được trạng thái Quay hẳn ra sau!"

# Kịch bản 4: Thí sinh quay mặt trở lại camera
estimator.reset_state()
pitch, yaw, viol = estimator.process(sample_rgb)
print(f"4. Quay mặt trở lại camera   : Pitch={pitch}°, Yaw={yaw}° -> Trạng thái: {viol or 'Bình thường'}")
print("\n✅ TẤT CẢ CÁC CA KIỂM THỬ HYSTERESIS ĐÃ VƯỢT QUA CHUẨN XÁC!")


---
## 🔴 Bước 8: Demo Tương Tác Webcam Thời Gian Thực (Interactive Real-Time)
Chạy ô code bên dưới để mở cửa sổ Webcam tương tác trực tiếp với giảng viên:
* Hệ thống sẽ vẽ trực tiếp **3 trục tọa độ 3D**, **HUD tiếng Việt**, góc **Pitch/Yaw** và **Trạng thái vi phạm**.
* Bạn có thể thử nghiệm quay đầu sang trái, phải, cúi mặt, ngửa mặt hoặc ngoái hẳn ra sau để thầy trực tiếp quan sát.
* **Nhấn phím `q` trên cửa sổ OpenCV để đóng camera.**


In [ ]:
# BƯỚC 8: VÒNG LẶP WEBCAM TRỰC TIẾP (NHẤN 'q' ĐỂ THOÁT)
cap = cv2.VideoCapture(cfg.camera.device_id)
if not cap.isOpened():
    print("❌ Không thể kết nối Webcam vật lý. Hãy chắc chắn không có ứng dụng nào khác đang chiếm camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, cfg.camera.width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cfg.camera.height)
    estimator = HeadPoseEstimator(cfg.camera.width, cfg.camera.height)
    
    print("🚀 Đang mở cửa sổ Webcam Demo... Nhấn phím 'q' trên cửa sổ để tắt.")
    
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break
            
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # 1. Tính toán góc đầu & vi phạm
        pitch, yaw, violation = estimator.process(frame_rgb)
        
        # 2. Vẽ thông tin HUD
        color = (0, 255, 0)
        status_str = "BINH THUONG - HOP LE"
        if violation:
            color = (0, 0, 255) if "sau" in violation.lower() or "người" in violation.lower() else (0, 255, 255)
            status_str = f"CHU Y: {violation.upper()}"
            
        # Vẽ thanh header
        cv2.rectangle(frame, (10, 10), (630, 85), (20, 20, 20), -1)
        cv2.rectangle(frame, (10, 10), (630, 85), color, 2)
        cv2.putText(frame, f"STATUS: {status_str}", (25, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        cv2.putText(frame, f"Pitch: {pitch:+.1f} deg | Yaw: {yaw:+.1f} deg", (25, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        cv2.imshow("ProctorAI - Head Pose Demo (Nhan 'q' de thoat)", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()
    estimator.close()
    print("👋 Đã đóng camera demo thành công!")
